# ChatPromptTemplate 고급 활용 — MessagesPlaceholder와 멀티턴 대화

Ch01에서 LLM 호출을 배웠다면, 이제 프롬프트를 체계적으로 관리하는 법을 배워요.
`01-PromptTemplate`에서 기본 `ChatPromptTemplate` 사용법을 익혔다면, 이 노트북에서는 **대화 이력**을 동적으로 다루는 방법을 배워요.

실제 챗봇은 이전 대화를 기억해야 해요. `MessagesPlaceholder`(메시지 플레이스홀더)가 그 역할을 담당합니다.

## 학습 목표

- `MessagesPlaceholder`로 가변 길이 대화 이력을 프롬프트에 삽입할 수 있어요
- `HumanMessage` / `AIMessage`로 대화 이력 리스트를 직접 구성할 수 있어요
- 멀티턴(multi-turn) 대화 시스템을 설계하고 이력 길이를 제한하는 방법을 알 수 있어요
- 공통 시스템 메시지를 재사용해 여러 프롬프트를 효율적으로 관리할 수 있어요

## 사전 지식

- `01-PromptTemplate` 노트북의 `ChatPromptTemplate` 기본 사용법
- LCEL 파이프라인 (`|` 연산자) 기초

---

> **ChatPromptTemplate 메시지 구성도** — 아래 다이어그램은 System, Human, AI 메시지와 대화 이력이 어떻게 조합되어 LLM에 전달되는지 보여줘요.

```mermaid
flowchart TD
    SYS["SystemMessage<br/>'당신은 AI 어시스턴트입니다.'"]:::process
    HIST["MessagesPlaceholder<br/>chat_history<br/>HumanMessage, AIMessage, ...<br/>이전 대화 이력 (가변 길이)"]:::storage
    HUM["HumanMessage<br/>현재 질문 {question}"]:::input
    LIMIT["이력 길이 제한<br/>최근 N개 메시지만 전달<br/>토큰 비용 제어"]:::error
    FINAL["ChatPromptTemplate<br/>메시지 배열 조립"]:::process
    LLM["LLM"]:::external
    RES["AIMessage<br/>응답 (맥락 반영)"]:::output

    SYS --> FINAL
    HIST -->|"전체 또는 최근 N개"| LIMIT
    LIMIT --> FINAL
    HUM --> FINAL
    FINAL --> LLM
    LLM --> RES

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

In [1]:
# 필수 라이브러리 import
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# 모델 초기화
# temperature=0: 일관된 출력을 위해 설정
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. MessagesPlaceholder — 동적 대화 이력 삽입

`MessagesPlaceholder`(메시지 플레이스홀더)는 가변 길이의 메시지 리스트를 프롬프트에 삽입할 수 있게 해줘요.

> 💡 **실무 팁:** 대화 이력이 길어질수록 토큰 사용량이 늘어나요. 프로덕션 챗봇에서는 `chat_history[-10:]`처럼 최근 N개만 전달하거나, 오래된 대화를 요약해서 압축하는 전략을 사용해요.

### 언제 사용할까요?

- 멀티턴 대화 시스템 (챗봇, 대화형 에이전트)
- 이전 대화 내용을 기억해야 하는 경우
- 컨텍스트를 유지하는 QA 시스템

### 주요 메시지 타입

| 클래스 | 역할 | 사용 시점 |
|--------|------|----------|
| `SystemMessage` | LLM 행동 방식·규칙 정의 | 프롬프트 맨 앞 |
| `HumanMessage` | 사용자 발화 | 대화 이력에 추가 |
| `AIMessage` | LLM 이전 응답 | 대화 이력에 추가 |

> 🔑 **핵심 개념**: `MessagesPlaceholder`는 `{변수명}` 플레이스홀더의 메시지 리스트 버전이에요. `invoke({"chat_history": [메시지들]})`처럼 메시지 리스트를 넘기면 해당 위치에 그대로 삽입됩니다. 이것이 모든 LangChain 챗봇의 메모리 동작 원리예요.

> 🎯 **강의 포인트**: `chat_history=[]`로 시작해서 매 턴마다 `HumanMessage + AIMessage` 쌍을 추가하는 패턴을 코드로 직접 보여주세요. 이 수동 관리 방식을 먼저 이해해야 이후 LangChain의 자동 메모리 클래스(`ConversationBufferMemory` 등)가 왜 편리한지 체감할 수 있어요.

In [2]:
# 시나리오: 이전 대화 내용을 기억하는 챗봇

# 1단계: MessagesPlaceholder를 포함한 ChatPromptTemplate 정의
# MessagesPlaceholder: 가변 길이의 메시지 리스트를 프롬프트 중간에 삽입하는 플레이스홀더
#   - variable_name: invoke() 호출 시 전달할 딕셔너리 키 이름
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다. 이전 대화 맥락을 고려하여 답변합니다."),
    MessagesPlaceholder(variable_name="chat_history"),  # 대화 이력이 들어갈 자리
    ("human", "{question}")  # 현재 질문
])

# 2단계: 체인 구성
chain = prompt | model | StrOutputParser()

# 3단계: 첫 번째 대화 (이력 없음)
result1 = chain.invoke({
    "chat_history": [],  # 빈 리스트: 아직 이전 대화 없음
    "question": "제 이름은 철수입니다. 기억해주세요."
})

print("=" * 60)
print("첫 번째 대화:")
print("=" * 60)
print(result1)

# 4단계: 두 번째 대화 (이력 포함)
# HumanMessage / AIMessage: 이전 대화를 메시지 객체로 표현
chat_history = [
    HumanMessage(content="제 이름은 철수입니다. 기억해주세요."),
    AIMessage(content=result1)
]

result2 = chain.invoke({
    "chat_history": chat_history,
    "question": "제 이름이 뭐였죠?"
})

print("\n" + "=" * 60)
print("두 번째 대화 (이름 기억 확인):")
print("=" * 60)
print(result2)

첫 번째 대화:
안녕하세요, 철수님! 반갑습니다. 어떻게 도와드릴까요?

두 번째 대화 (이름 기억 확인):
철수님이시죠! 다른 질문이나 도움이 필요하시면 말씀해 주세요.


## 2. 복잡한 대화 시스템 — 고객 지원 챗봇

실제 업무에서 사용할 수 있는 멀티턴 대화 시스템을 구축해볼게요.

> ⚠️ **자주 하는 실수**: 챗봇에서 `chat_history`를 관리할 때는 매 턴마다 `HumanMessage`와 `AIMessage`를 쌍으로 추가해야 해요. 하나만 추가하면 역할 순서가 어긋나 LLM이 혼란스러워할 수 있어요.

> 🎯 **강의 포인트**: `chat_history.extend([HumanMessage(content=msg), AIMessage(content=response)])` 패턴이 핵심이에요. `append()`를 두 번 쓰는 것과 `extend()`를 한 번 쓰는 것이 동일하지만, extend로 쌍을 한번에 추가하면 실수가 줄어요. 이 패턴을 처음부터 습관화하도록 가르치세요.

> 💡 **실무 팁**: 실제 프로덕션 챗봇에서는 `chat_history`를 DB에 저장하고 불러와야 해요. Redis, DynamoDB 등에 세션별로 저장하는 패턴이 일반적이며, LangChain의 `RedisChatMessageHistory` 같은 클래스를 활용할 수 있어요.

In [3]:
# 시나리오: 제품 문의를 처리하는 고객 지원 챗봇

# 1단계: 고객 지원 챗봇 프롬프트 정의
support_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 전자제품 쇼핑몰의 고객 지원 AI입니다.
    
다음 규칙을 따라주세요:
- 항상 정중하고 친절하게 응답
- 이전 대화 내용을 기억하고 맥락에 맞게 답변
- 제품 정보, 배송, 환불 문의에 대응
- 구체적인 정보가 필요하면 질문
- 짧고 명확하게 답변"""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{message}")
])

# 2단계: 체인 구성
support_chain = support_prompt | model | StrOutputParser()

# 3단계: 대화 시뮬레이션
# chat_history: 매 턴마다 HumanMessage + AIMessage 쌍을 추가해 대화 맥락 유지
chat_history = []

# 대화 1
message1 = "무선 이어폰 추천해주세요."
response1 = support_chain.invoke({"history": chat_history, "message": message1})
print("=" * 60)
print(f"고객: {message1}")
print(f"AI: {response1}")
chat_history.extend([
    HumanMessage(content=message1),
    AIMessage(content=response1)
])

# 대화 2
message2 = "가격대는 10만원 이하로 원해요."
response2 = support_chain.invoke({"history": chat_history, "message": message2})
print("\n" + "=" * 60)
print(f"고객: {message2}")
print(f"AI: {response2}")
chat_history.extend([
    HumanMessage(content=message2),
    AIMessage(content=response2)
])

# 대화 3
message3 = "배송은 얼마나 걸리나요?"
response3 = support_chain.invoke({"history": chat_history, "message": message3})
print("\n" + "=" * 60)
print(f"고객: {message3}")
print(f"AI: {response3}")

고객: 무선 이어폰 추천해주세요.
AI: 무선 이어폰을 추천해드리겠습니다! 사용 용도에 따라 다르지만, 다음 몇 가지 모델을 고려해보세요:

1. **애플 에어팟 프로**: 뛰어난 음질과 노이즈 캔슬링 기능이 특징입니다.
2. **소니 WF-1000XM4**: 탁월한 음질과 긴 배터리 수명을 자랑합니다.
3. **삼성 갤럭시 버즈2**: 편안한 착용감과 좋은 음질을 제공합니다.
4. **안커 사운드코어 라이프 P3**: 가성비가 뛰어나며, 다양한 기능이 포함되어 있습니다.

어떤 용도로 사용하실 예정인지 알려주시면 더 구체적인 추천을 드릴 수 있습니다!

고객: 가격대는 10만원 이하로 원해요.
AI: 10만원 이하의 무선 이어폰 중에서 추천드릴 만한 제품은 다음과 같습니다:

1. **안커 사운드코어 라이프 P2**: 좋은 음질과 편안한 착용감, 긴 배터리 수명을 제공합니다.
2. **샤오미 레드미 에어닷 3**: 가성비가 뛰어나고 기본적인 기능을 잘 갖추고 있습니다.
3. **JBL TUNE 125TWS**: JBL의 음질을 경험할 수 있으며, 착용감도 좋습니다.
4. **삼성 갤럭시 버즈 라이브**: 독특한 디자인과 좋은 음질을 제공하며, 가격대도 적당합니다.

이 중에서 어떤 제품이 마음에 드시나요? 추가 정보가 필요하시면 말씀해 주세요!

고객: 배송은 얼마나 걸리나요?
AI: 배송 기간은 주문하신 제품의 재고 상황과 배송 지역에 따라 다를 수 있습니다. 일반적으로는 다음과 같은 기준이 있습니다:

- **재고가 있는 경우**: 1~3일 이내에 배송됩니다.
- **재고가 없는 경우**: 3~7일 정도 소요될 수 있습니다.

정확한 배송 기간은 주문 시 확인할 수 있으며, 특정 지역에 따라 다를 수 있습니다. 추가로 궁금한 점이 있으시면 말씀해 주세요!


## 3. 템플릿 재사용 — 공통 시스템 메시지 공유하기

번역·요약·교정처럼 역할만 다르고 지침 구조가 같은 작업이라면, `SystemMessagePromptTemplate`을 하나 정의해두고 여러 프롬프트에서 공유할 수 있어요.

> 💡 **실무 팁:** 공통 시스템 메시지를 변수로 분리해두면, 회사 정책이 바뀌었을 때 한 곳만 수정하면 모든 체인에 반영돼요. 유지보수 비용이 크게 줄어들어요.

> 🎯 **강의 포인트**: 소프트웨어 공학의 DRY(Don't Repeat Yourself) 원칙이 프롬프트 엔지니어링에도 그대로 적용돼요. 공통 지침(응답 언어, 톤, 형식)을 하나의 `SystemMessagePromptTemplate`에 모아두고 여러 체인에서 공유하면 일관성이 보장되고 수정도 쉬워집니다.

> ⚠️ **자주 하는 실수**: `common_system`을 여러 프롬프트에서 재사용할 때, 한 프롬프트에서 `common_system`을 수정(mutate)하면 다른 프롬프트에도 영향을 줄 수 있어요. 객체를 공유하는 대신 `partial()`로 파생 객체를 만드는 패턴이 더 안전합니다.

In [4]:
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

# 시나리오: 여러 프롬프트에서 동일한 시스템 메시지 재사용

# 1단계: 공통 시스템 메시지 템플릿 정의
# SystemMessagePromptTemplate: 시스템 역할을 변수로 분리하여 재사용 가능하게 구성
common_system = SystemMessagePromptTemplate.from_template(
    """당신은 {role}입니다.
    
지침:
- {style} 스타일로 작성
- {length} 단어 이내로 답변
- 전문적이고 정확하게"""
)

# 2단계: 번역 프롬프트에 재사용
# HumanMessagePromptTemplate: 사용자 질문 부분을 독립적으로 정의
translation_human = HumanMessagePromptTemplate.from_template(
    "{source_lang}을 {target_lang}로 번역: {text}"
)

translation_prompt = ChatPromptTemplate.from_messages([
    common_system,
    translation_human
])

# 3단계: 요약 프롬프트에 재사용
# 동일한 common_system을 재사용 → 회사 정책 변경 시 한 곳만 수정하면 됨
summary_human = HumanMessagePromptTemplate.from_template(
    "다음 텍스트를 요약: {text}"
)

summary_prompt = ChatPromptTemplate.from_messages([
    common_system,
    summary_human
])

# 4단계: 번역 실행
translation_chain = translation_prompt | model | StrOutputParser()
result_translation = translation_chain.invoke({
    "role": "전문 번역가",
    "style": "자연스럽고 유창한",
    "length": "50",
    "source_lang": "한국어",
    "target_lang": "영어",
    "text": "인공지능은 현대 기술의 핵심입니다."
})

print("=" * 60)
print("번역 결과:")
print("=" * 60)
print(result_translation)

# 5단계: 요약 실행
summary_chain = summary_prompt | model | StrOutputParser()
result_summary = summary_chain.invoke({
    "role": "콘텐츠 에디터",
    "style": "간결하고 핵심적인",
    "length": "30",
    "text": "머신러닝은 데이터로부터 패턴을 학습하는 인공지능의 한 분야입니다. 지도학습, 비지도학습, 강화학습 등 다양한 방법론이 있으며, 이미지 인식, 자연어 처리, 추천 시스템 등에 활용됩니다."
})

print("\n" + "=" * 60)
print("요약 결과:")
print("=" * 60)
print(result_summary)

print("\n💡 동일한 시스템 메시지를 여러 프롬프트에서 재사용했습니다!")

번역 결과:
Artificial intelligence is at the core of modern technology.

요약 결과:
머신러닝은 데이터에서 패턴을 학습하는 AI 분야로, 지도학습, 비지도학습, 강화학습을 포함하며 이미지 인식, 자연어 처리, 추천 시스템에 활용됩니다.

💡 동일한 시스템 메시지를 여러 프롬프트에서 재사용했습니다!


## 💡 핵심 정리

### MessagesPlaceholder 사용법

```python
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# MessagesPlaceholder 포함 프롬프트
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# 대화 이력 전달
chain.invoke({
    "chat_history": [
        HumanMessage(content="안녕하세요"),
        AIMessage(content="안녕하세요! 무엇을 도와드릴까요?")
    ],
    "question": "날씨 알려주세요"
})
```

### 주요 메시지 타입

| 메시지 타입 | 설명 | 사용 시점 |
|-----------|------|-----------|
| `SystemMessage` | 시스템 지침 | LLM 역할/규칙 정의 |
| `HumanMessage` | 사용자 메시지 | 사용자 질문/요청 |
| `AIMessage` | AI 응답 | LLM의 이전 응답 |

### 활용 시나리오

**언제 사용하면 좋을까요?**
- ✅ 멀티턴 대화 시스템 (챗봇, 대화형 에이전트)
- ✅ 이전 대화를 기억해야 하는 경우
- ✅ 컨텍스트가 중요한 QA 시스템
- ✅ 고객 지원, 상담 챗봇

**주의사항**:
- ⚠️ 대화 이력이 길어지면 토큰 사용량 증가 → 비용 상승
- ⚠️ 적절한 이력 길이 제한 필요 (최근 N개만 유지)
- ⚠️ 메시지 타입을 정확히 지정 (HumanMessage, AIMessage)

### 대화 이력 관리 팁

```python
# 최근 N개 메시지만 유지
MAX_HISTORY = 10
chat_history = chat_history[-MAX_HISTORY:]

# 토큰 수로 제한 (tiktoken 사용)
# 토큰 수가 제한을 초과하면 오래된 메시지부터 제거
```


## 연습 문제

직접 해보면서 학습 내용을 정리해봅시다!

### 연습 1: 멀티턴 기술 면접 시뮬레이터

**과제**: `MessagesPlaceholder`를 사용하여 기술 면접을 시뮬레이션하는 대화 시스템을 만드세요.

**요구사항**:
- `ChatPromptTemplate` + `MessagesPlaceholder` 사용
- 시스템 메시지: 면접관 역할 정의 (Python 시니어 개발자 면접관)
- 3턴 이상의 대화를 시뮬레이션 (질문 -> 답변 -> 후속 질문)
- 대화 이력(`chat_history`)을 매 턴마다 업데이트

**힌트**: 면접관이 이전 답변을 바탕으로 후속 질문을 하도록 대화 이력을 활용하세요.

In [5]:
# ============================================================
# TODO: MessagesPlaceholder를 사용하여 멀티턴 기술 면접 시뮬레이터를 만드세요
# 힌트: ChatPromptTemplate.from_messages([("system", ...), MessagesPlaceholder(...), ("human", ...)])로 구성하고
#       매 턴마다 chat_history에 HumanMessage + AIMessage 쌍을 추가하여 이전 답변을 유지하세요
# 예상 결과: 면접관이 이전 답변을 바탕으로 후속 질문을 하는 3턴 이상의 대화
# ============================================================

# 여기에 코드를 작성하세요

# interview_prompt = ChatPromptTemplate.from_messages([
#     ("system", "..."),
#     MessagesPlaceholder(variable_name="chat_history"),
#     ("human", "{answer}")
# ])

### 연습 1 풀이

In [6]:
# 연습 1 풀이: 멀티턴 기술 면접 시뮬레이터

# 1단계: 면접관 프롬프트 정의
interview_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 Python 시니어 개발자 면접관입니다.

규칙:
- 면접자의 답변을 평가하고 후속 질문을 합니다
- 답변이 부족하면 힌트를 주며 더 깊은 답변을 유도합니다
- 답변이 충분하면 다음 주제로 넘어갑니다
- 한 번에 하나의 질문만 합니다
- 친절하지만 전문적인 어조를 유지합니다"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{answer}")
])

# 2단계: 체인 구성
interview_chain = interview_prompt | model | StrOutputParser()

# 3단계: 멀티턴 대화 시뮬레이션
chat_history = []

# 턴 1: 면접 시작
answer1 = "안녕하세요, 면접에 참석하게 되어 기쁩니다. Python 개발 경력 3년차입니다."
response1 = interview_chain.invoke({
    "chat_history": chat_history,
    "answer": answer1
})
print("=" * 60)
print(f"면접자: {answer1}")
print(f"면접관: {response1}")
chat_history.extend([
    HumanMessage(content=answer1),
    AIMessage(content=response1)
])

# 턴 2: 기술 질문에 답변
answer2 = "리스트는 변경 가능(mutable)하고 대괄호를 사용합니다. 튜플은 변경 불가능(immutable)하고 소괄호를 사용합니다. 튜플이 메모리 효율이 더 좋고, 딕셔너리의 키로 사용할 수 있습니다."
response2 = interview_chain.invoke({
    "chat_history": chat_history,
    "answer": answer2
})
print("\n" + "=" * 60)
print(f"면접자: {answer2}")
print(f"면접관: {response2}")
chat_history.extend([
    HumanMessage(content=answer2),
    AIMessage(content=response2)
])

# 턴 3: 후속 질문에 답변
answer3 = "데코레이터는 함수를 감싸서 기능을 추가하는 패턴입니다. @login_required 같은 인증 체크나 @cache 같은 캐싱에 자주 사용합니다. 함수를 인자로 받아 새 함수를 반환하는 고차 함수입니다."
response3 = interview_chain.invoke({
    "chat_history": chat_history,
    "answer": answer3
})
print("\n" + "=" * 60)
print(f"면접자: {answer3}")
print(f"면접관: {response3}")

면접자: 안녕하세요, 면접에 참석하게 되어 기쁩니다. Python 개발 경력 3년차입니다.
면접관: 안녕하세요! 면접에 참석해 주셔서 감사합니다. Python 개발 경력이 3년이라니 인상적입니다. 먼저, Python을 사용하여 진행한 프로젝트 중 가장 기억에 남는 프로젝트에 대해 말씀해 주실 수 있나요? 그 프로젝트에서 어떤 역할을 하셨고, 어떤 기술을 사용하셨는지 궁금합니다.

면접자: 리스트는 변경 가능(mutable)하고 대괄호를 사용합니다. 튜플은 변경 불가능(immutable)하고 소괄호를 사용합니다. 튜플이 메모리 효율이 더 좋고, 딕셔너리의 키로 사용할 수 있습니다.
면접관: 좋은 설명 감사합니다! 리스트와 튜플의 차이점을 잘 정리해 주셨네요. 추가로, 리스트와 튜플을 선택할 때 어떤 상황에서 각각을 사용하는 것이 더 적합하다고 생각하시나요? 예를 들어, 특정한 사용 사례나 성능 측면에서의 차이를 말씀해 주실 수 있을까요?

면접자: 데코레이터는 함수를 감싸서 기능을 추가하는 패턴입니다. @login_required 같은 인증 체크나 @cache 같은 캐싱에 자주 사용합니다. 함수를 인자로 받아 새 함수를 반환하는 고차 함수입니다.
면접관: 데코레이터에 대한 설명이 매우 좋습니다! 데코레이터의 기본 개념과 사용 예시를 잘 설명해 주셨네요. 

그렇다면, 데코레이터를 구현할 때 주의해야 할 점이나, 데코레이터를 사용할 때 발생할 수 있는 문제점에 대해 말씀해 주실 수 있나요? 예를 들어, 데코레이터가 원래 함수의 메타데이터를 손상시킬 수 있는 경우에 대해 어떻게 처리할 수 있을까요?


### 연습 2: 재사용 가능한 다목적 분석 시스템

**과제**: 공통 시스템 메시지 템플릿을 재사용하여 여러 분석 작업(감정 분석, 키워드 추출)을 수행하는 시스템을 만드세요.

**요구사항**:
- `SystemMessagePromptTemplate`과 `HumanMessagePromptTemplate`을 사용
- 공통 시스템 메시지를 정의하고 두 가지 이상의 분석 프롬프트에서 재사용
- 변수: `role`, `output_format`, `text`
- 감정 분석 체인과 키워드 추출 체인을 각각 구성하여 실행

**힌트**: 공통 시스템 메시지에서 `role`과 `output_format`을 변수로 사용하면 다양한 분석에 재사용할 수 있습니다.

In [7]:
# ============================================================
# TODO: 공통 SystemMessagePromptTemplate을 재사용하여 감정 분석과 키워드 추출 체인을 만드세요
# 힌트: SystemMessagePromptTemplate.from_template(...)으로 role과 output_format을 변수로 갖는
#       공통 시스템 메시지를 정의하고, 두 프롬프트에서 동일한 객체를 재사용하세요
# 예상 결과: 같은 텍스트에 대해 감정 분석 결과와 키워드 추출 결과를 각각 출력
# ============================================================

# 여기에 코드를 작성하세요

# common_system = SystemMessagePromptTemplate.from_template(
#     "당신은 {role}입니다. 결과를 {output_format} 형식으로 출력합니다."
# )
#
# analysis_human = HumanMessagePromptTemplate.from_template(
#     "다음 텍스트를 분석해주세요:\n{text}"
# )

### 연습 2 풀이

In [8]:
# 연습 2 풀이: 재사용 가능한 다목적 분석 시스템

from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

# 1단계: 공통 시스템 메시지 템플릿 정의
common_system = SystemMessagePromptTemplate.from_template(
    """당신은 {role}입니다.

분석 결과를 다음 형식으로 출력합니다:
{output_format}

간결하고 정확하게 분석해주세요."""
)

# 2단계: 공통 사용자 메시지 템플릿 정의
analysis_human = HumanMessagePromptTemplate.from_template(
    "다음 텍스트를 분석해주세요:\n{text}"
)

# 3단계: 감정 분석 프롬프트 구성 (공통 시스템 메시지 재사용)
sentiment_prompt = ChatPromptTemplate.from_messages([
    common_system,
    analysis_human
])

sentiment_chain = sentiment_prompt | model | StrOutputParser()

# 4단계: 키워드 추출 프롬프트 구성 (같은 공통 시스템 메시지 재사용)
keyword_prompt = ChatPromptTemplate.from_messages([
    common_system,
    analysis_human
])

keyword_chain = keyword_prompt | model | StrOutputParser()

# 5단계: 분석할 텍스트
sample_text = """최근 출시된 AI 비서 서비스가 큰 인기를 끌고 있습니다. 
사용자들은 자연스러운 대화 능력과 빠른 응답 속도에 만족감을 표현하고 있으나, 
가끔 부정확한 정보를 제공한다는 점에서 아쉬움을 토로하는 의견도 있습니다."""

# 감정 분석 실행
result_sentiment = sentiment_chain.invoke({
    "role": "감정 분석 전문가",
    "output_format": "- 전체 감정: (긍정/부정/중립/복합)\n- 긍정 요소: (목록)\n- 부정 요소: (목록)\n- 감정 강도: (1~5점)",
    "text": sample_text
})

print("=" * 60)
print("감정 분석 결과:")
print("=" * 60)
print(result_sentiment)

# 키워드 추출 실행
result_keywords = keyword_chain.invoke({
    "role": "키워드 추출 전문가",
    "output_format": "- 핵심 키워드: (최대 5개, 쉼표 구분)\n- 주제 분류: (카테고리)\n- 요약: (한 문장)",
    "text": sample_text
})

print("\n" + "=" * 60)
print("키워드 추출 결과:")
print("=" * 60)
print(result_keywords)

print("\n" + "=" * 60)
print("동일한 공통 시스템 메시지를 두 가지 분석에 재사용했습니다!")
print("=" * 60)

감정 분석 결과:
- 전체 감정: 복합
- 긍정 요소: (큰 인기, 자연스러운 대화 능력, 빠른 응답 속도, 만족감)
- 부정 요소: (부정확한 정보 제공, 아쉬움)
- 감정 강도: 3점

키워드 추출 결과:
- 핵심 키워드: AI 비서, 인기, 자연스러운 대화, 빠른 응답, 부정확한 정보
- 주제 분류: 기술/AI
- 요약: 최근 출시된 AI 비서 서비스는 사용자들에게 높은 만족도를 주지만, 부정확한 정보 제공에 대한 아쉬움도 존재합니다.

동일한 공통 시스템 메시지를 두 가지 분석에 재사용했습니다!


### 연습 3: 대화 이력 길이 제한이 있는 챗봇

**과제**: 대화 이력을 최근 N개 메시지로 제한하는 챗봇을 만드세요.

**요구사항**:
- `MessagesPlaceholder`를 사용한 대화형 시스템
- 대화 이력을 최근 4개 메시지(2턴)로 제한
- 5턴 이상의 대화를 시뮬레이션하여 이력 제한이 동작하는지 확인
- 시스템 메시지: 여행 가이드 AI 역할

**힌트**: `chat_history[-MAX_MESSAGES:]` 슬라이싱을 사용하면 최근 메시지만 유지할 수 있습니다.

In [9]:
# ============================================================
# TODO: 대화 이력을 최근 4개 메시지로 제한하는 여행 가이드 챗봇을 만드세요
# 힌트: chat_history[-MAX_MESSAGES:]로 슬라이싱하여 최근 메시지만 LLM에 전달하고
#       전체 이력(chat_history)에는 모든 메시지를 계속 누적하세요
# 예상 결과: 5턴 이후 초기 대화 내용을 기억하지 못하는 이력 제한 동작 확인
# ============================================================

# 여기에 코드를 작성하세요

# MAX_MESSAGES = 4  # 최근 4개 메시지(2턴)만 유지
#
# travel_prompt = ChatPromptTemplate.from_messages([
#     ("system", "..."),
#     MessagesPlaceholder(variable_name="chat_history"),
#     ("human", "{question}")
# ])

### 연습 3 풀이

In [10]:
# 연습 3 풀이: 대화 이력 길이 제한이 있는 챗봇

# 대화 이력 최대 길이 설정 (메시지 개수 기준, 2턴 = 4개 메시지)
MAX_MESSAGES = 4

# 1단계: 여행 가이드 프롬프트 정의
travel_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 전문 여행 가이드 AI입니다.
    
규칙:
- 여행지 추천, 일정 계획, 맛집 추천 등에 답변합니다
- 이전 대화를 참고하여 맥락에 맞게 답변합니다
- 짧고 핵심적으로 답변합니다 (3문장 이내)"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# 2단계: 체인 구성
travel_chain = travel_prompt | model | StrOutputParser()

# 3단계: 대화 이력 관리 함수
def chat_with_limit(chain, chat_history, question, max_messages=MAX_MESSAGES):
    """대화 이력을 제한하면서 대화하는 함수"""
    # 최근 max_messages개만 전달
    limited_history = chat_history[-max_messages:]
    
    response = chain.invoke({
        "chat_history": limited_history,
        "question": question
    })
    
    # 전체 이력에는 모두 저장 (나중에 참고용)
    chat_history.extend([
        HumanMessage(content=question),
        AIMessage(content=response)
    ])
    
    return response

# 4단계: 5턴 대화 시뮬레이션
chat_history = []
questions = [
    "제주도 3박 4일 여행을 계획하고 있어요.",
    "첫째 날 일정을 추천해주세요.",
    "둘째 날은 해변 위주로 가고 싶어요.",
    "맛집도 추천해주세요.",
    "제가 처음에 어디로 여행 간다고 했죠?",  # 이력 제한 확인용 질문
]

for i, question in enumerate(questions, 1):
    print(f"{'=' * 60}")
    print(f"[턴 {i}] (전체 이력: {len(chat_history)}개, 전달 이력: {min(len(chat_history), MAX_MESSAGES)}개)")
    print(f"사용자: {question}")
    
    response = chat_with_limit(travel_chain, chat_history, question)
    print(f"AI: {response}")
    print()

print("=" * 60)
print(f"최종 전체 이력: {len(chat_history)}개 메시지")
print(f"매 턴마다 최근 {MAX_MESSAGES}개 메시지만 LLM에 전달됨")
print("마지막 질문에서 초기 대화 내용을 기억하지 못할 수 있습니다 (이력 제한 때문)")
print("=" * 60)

[턴 1] (전체 이력: 0개, 전달 이력: 0개)
사용자: 제주도 3박 4일 여행을 계획하고 있어요.
AI: 제주도 3박 4일 여행은 멋진 선택입니다! 첫째 날에는 한라산 등반 후 제주시의 동문시장에서 저녁을 즐기고, 둘째 날에는 성산일출봉과 우도 탐방을 추천합니다. 셋째 날에는 협재 해수욕장에서 여유를 즐기고, 넷째 날에는 제주 민속촌을 방문해 보세요.

[턴 2] (전체 이력: 2개, 전달 이력: 2개)
사용자: 첫째 날 일정을 추천해주세요.
AI: 첫째 날은 아침에 제주 도착 후, 한라산 성판악 코스를 선택해 등반을 시작하세요. 오후에는 제주시로 돌아와 동문시장에서 제주 특산물과 맛있는 해산물 요리를 즐기세요. 저녁에는 근처의 바다 전망 카페에서 여유롭게 하루를 마무리하는 것도 좋습니다.

[턴 3] (전체 이력: 4개, 전달 이력: 4개)
사용자: 둘째 날은 해변 위주로 가고 싶어요.
AI: 둘째 날은 성산일출봉에서 일출을 감상한 후, 근처의 섭지코지로 이동해 아름다운 해안 경치를 즐기세요. 오후에는 우도로 가서 해변에서 수영이나 자전거 타기를 추천합니다. 저녁에는 우도의 땅콩 아이스크림과 해산물 요리를 맛보세요!

[턴 4] (전체 이력: 6개, 전달 이력: 4개)
사용자: 맛집도 추천해주세요.
AI: 제주에서 추천할 만한 맛집은 다음과 같습니다: 

1. **돈사돈** - 제주 흑돼지 구이로 유명한 곳입니다.
2. **해녀의 집** - 신선한 해산물 요리를 즐길 수 있는 곳으로, 해물뚝배기가 인기입니다.
3. **제주도민속자연사박물관 근처의 고기국수집** - 제주 전통 고기국수를 맛볼 수 있는 아늑한 식당입니다. 

각 맛집에서 제주 특산물을 꼭 경험해보세요!

[턴 5] (전체 이력: 8개, 전달 이력: 4개)
사용자: 제가 처음에 어디로 여행 간다고 했죠?
AI: 처음에 제주도로 여행 간다고 말씀하셨습니다. 제주도에서의 여행 계획에 대해 계속 도와드릴게요!

최종 전체 이력: 10개 메시지
매 턴마다 최근 4개 메시지만 LLM에 전달됨
마지막 

## 마무리 — 이번 노트북에서 배운 것

| 개념 | 핵심 클래스 | 언제 쓰나요? |
|------|------------|-------------|
| 동적 대화 이력 삽입 | `MessagesPlaceholder` | 멀티턴 챗봇, 이전 대화를 기억해야 할 때 |
| 대화 이력 구성 | `HumanMessage`, `AIMessage` | 이전 발화를 메시지 객체로 표현할 때 |
| 이력 길이 제한 | `history[-N:]` 슬라이싱 | 토큰 비용을 제어하고 싶을 때 |
| 템플릿 재사용 | `SystemMessagePromptTemplate` | 공통 지침을 여러 프롬프트에서 공유할 때 |

`MessagesPlaceholder`는 단순한 프롬프트를 진짜 대화 시스템으로 끌어올리는 핵심 도구예요. 대화 이력을 어떻게 관리하느냐에 따라 챗봇의 품질이 크게 달라집니다.

---

**다음 챕터 예고:** `03_OutputParser` — LLM이 반환하는 자유 형식 텍스트를 JSON, 리스트 등 구조화된 데이터로 파싱하는 방법을 배워요. 프롬프트 → LLM → 파싱까지 이어지는 파이프라인이 완성됩니다.